In [74]:
import os
from pathlib import Path
import re
from glider import GoProVideoMetadata, GoProGPX

def parse_gopro_filename(filename):
    m = re.match(r"^[A-Z]{2}(\d{2})(\d{4})\.MP4$", filename, re.IGNORECASE)
    if not m:
        raise ValueError(f"Invalid GoPro filename: {filename}")
    chapter = int(m.group(1))
    video_id = int(m.group(2))
    return chapter, video_id


In [75]:
base_path = Path("/home/sam/Videos")
videos = {}
for file in os.listdir(base_path):
    path_to_file = base_path / file
    try:
        chapter, video_id = parse_gopro_filename(file)
        junks = videos.get(video_id, list())
        junks.append((
            chapter,
            path_to_file,
            GoProVideoMetadata.from_video(path_to_file),
            GoProGPX(path_to_file)
        ))
    except ValueError:
        continue
    videos[video_id] = junks

for x in videos.values():
    x.sort(key = lambda x: x[0])


In [105]:
junk = videos[110]

GOPRO_FPS = 59.94005994005994
from datetime import timedelta
junk[0]

(1,
 PosixPath('/home/sam/Videos/GX010110.MP4'),
 GoProVideoMetadata(creation_time=datetime.datetime(2026, 5, 9, 13, 56, 45, tzinfo=datetime.timezone.utc), number_of_frames=42720, duration_seconds=712.712, fps=59.94005994005994),
 <glider.GoProGPX at 0x791f1d157ed0>)

In [90]:
last_trackpoint = junk[-1][3].track[-1]
last_trackpoint

GPXTrackPoint(47.2959057, 8.9200801, elevation=np.float64(683.94), time=datetime.datetime(2026, 5, 9, 12, 9, 6, 877780), symbol='Square', position_dilution=np.float64(1.53), speed=np.float64(0.13))

In [104]:
summed_framecount = sum(j[2].number_of_frames for j in junk)
video_duration = timedelta(seconds=summed_framecount / GOPRO_FPS)
video_duration

video_start_time = last_trackpoint.time - video_duration
video_start_time

datetime.datetime(2026, 5, 9, 11, 56, 47, 539180)

In [86]:
import pandas as pd

pd.DataFrame([
    {"JunkId": j[0],
     "Path": j[1],
     "CreatedUtc": j[2].creation_time,
     "Frames": j[2].number_of_frames,
     "Duration": j[2].duration_seconds,} for j in junk
])


,JunkId,Path,CreatedUtc,Frames,Duration
0,1,/home/sam/Videos/GX010110.MP4,2026-05-09 13:56:45+00:00,42720,712.7120
1,2,/home/sam/Videos/GX020110.MP4,2026-05-09 13:56:45+00:00,1596,26.6266


GoProVideoMetadata(creation_time=datetime.datetime(2026, 5, 9, 13, 19, 1, tzinfo=datetime.timezone.utc), number_of_frames=42720, duration_seconds=712.712, fps=59.94005994005994)